## Issue #7, Notebook 7.1: Consumer Expenditure Survey pull (via FRED)

Series: `CXUAPPARELLB0101M`, Expenditures: Apparel and Services: All Consumer Units (BLS Consumer Expenditure Survey, distributed via FRED). Annual, USD, not seasonally adjusted.

Feeds the hook-stat context and Chart 2's trend-chasing spend trajectory.

## 1. Setup

In [1]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")

RAW_DIR = "../../../data/raw/cycle 2/issue-07"
PROCESSED_DIR = "../../../data/processed/cycle 2/issue-07"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 2. Load brand palette

In [5]:
# load brand palette
with open("../../../brand/palette.json") as f:
    palette_data = json.load(f)

palette = {c["name"]: c["hex"] for c in palette_data["palette"]}

COCOA = palette["Cocoa"]
DUSTY = palette["Dusty"]
PEACH = palette["Peach"]
CREAM = palette["Cream"]
SAGE  = palette["Sage"]

## 3. Extract and analyze data

### 3.1 Pull the series from FRED

In [6]:
SERIES_ID = "CXUAPPARELLB0101M"

url = "https://api.stlouisfed.org/fred/series/observations"
params = {
    "series_id": SERIES_ID,
    "api_key": FRED_API_KEY,
    "file_type": "json",
}

resp = requests.get(url, params=params)
data = resp.json()

# Same defensive check as prior FRED pulls: FRED returns a JSON error object
# rather than an HTTP error on a bad request/series ID.
if "observations" not in data:
    print(data)  # inspect the actual error before going further
    raise KeyError("No 'observations' in response, see printed error above")

### 3.2 Save the raw response

In [7]:
with open(f"{RAW_DIR}/cxu_apparel_all_consumer_units.json", "w") as f:
    json.dump(data, f, indent=2)

### 3.3 Parse into a dataframe

In [8]:
df = pd.DataFrame(data["observations"])
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = df[["year", "value"]].rename(columns={"value": "apparel_spend_usd"})
df = df.dropna().sort_values("year").reset_index(drop=True)

df.tail(10)  # sanity check against confirmed 2020-2024 values:
# 2020: 1,434 | 2021: 1,754 | 2022: 1,945 | 2023: 2,041 | 2024: 2,001

,year,apparel_spend_usd
31,2015,1846
32,2016,1803
33,2017,1833
34,2018,1866
35,2019,1883
36,2020,1434
37,2021,1754
38,2022,1945
39,2023,2041
40,2024,2001


### 3.4 Save processed data

In [9]:
df.to_csv(f"{PROCESSED_DIR}/apparel_spend_annual.csv", index=False)